# HDB Rental Core Data Preprocessing

This notebook creates the core model tables in `data/processed/core/`.

Core preprocessing:

- preserve original row identifiers before filtering;
- remove raw exact training duplicates and audit duplicates again after normalization;
- standardize dates, categorical labels, block identifiers, and flat types;
- derive calendar, location-key, lease-age, and floor-area features;
- drop constant columns and exclude all auxiliary sources;
- retain one normalized `FLAT_MODEL` column so rare-category pooling can be fitted inside each model-training fold;
- preserve every test row and its original order for submission alignment.

Target extremes remain because they can be genuine market observations. No target-derived predictor is created.


In [1]:
from pathlib import Path
import json
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.neighbors import BallTree

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

RANDOM_STATE = 42
EARTH_RADIUS_KM = 6_371.0088
SAVE_OUTPUTS = True


def locate_project_root() -> Path:
    # Find the repository root whether the notebook starts in root or notebooks/.
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd.parent,
        cwd / "CS5228/project/cs5228-group6",
        cwd.parent / "cs5228-group6",
    ]
    for candidate in candidates:
        if (candidate / "data/train.csv").exists() and (candidate / "data/test.csv").exists():
            return candidate
    raise FileNotFoundError("Could not locate a project root containing data/train.csv and data/test.csv")


PROJECT_ROOT = locate_project_root()
DATA_DIR = PROJECT_ROOT / "data"
AUX_DIR = DATA_DIR / "auxiliary"
OUTPUT_DIR = DATA_DIR / "processed" / "core"

print("Project root: .")
print(f"Auxiliary data: {AUX_DIR.relative_to(PROJECT_ROOT)}")

Project root: .
Auxiliary data: data/auxiliary


## 1. Load raw data and enforce the data contract

The checks below fail early if columns change or if IDs cannot preserve the Kaggle test-row order.

In [2]:
train_raw = pd.read_csv(DATA_DIR / "train.csv")
test_raw = pd.read_csv(DATA_DIR / "test.csv")

expected_features = [
    "RENT_APPROVAL_DATE", "TOWN", "BLOCK", "STREET", "FLAT_TYPE",
    "FLAT_MODEL", "FLOOR_AREA_SQM", "FURNISHED", "LEASE_COMMENCE_DATE", "FEE",
]
target = "MONTHLY_RENT"

assert train_raw.columns.tolist() == expected_features + [target]
assert test_raw.columns.tolist() == expected_features
assert train_raw[target].notna().all()

train_raw.insert(0, "ROW_ID", np.arange(len(train_raw), dtype=np.int64))
test_raw.insert(0, "ROW_ID", np.arange(len(test_raw), dtype=np.int64))
raw_train_rows = len(train_raw)
train_raw_before_dedup = train_raw.copy()

raw_exact_mask = train_raw.drop(columns="ROW_ID").duplicated(keep="first")
raw_exact_duplicates_removed = int(raw_exact_mask.sum())
train_raw = train_raw.loc[~raw_exact_mask].copy()

audit = pd.DataFrame({
    "raw_rows": [raw_train_rows, len(test_raw)],
    "rows_after_train_dedup": [len(train_raw), len(test_raw)],
    "missing_cells": [int(train_raw.isna().sum().sum()), int(test_raw.isna().sum().sum())],
    "exact_duplicates_removed": [raw_exact_duplicates_removed, 0],
    "exact_test_rows_preserved": [0, int(test_raw.drop(columns="ROW_ID").duplicated().sum())],
}, index=["train", "test"])
display(audit)
print("Training policy: keep first exact row in original order; preserve every test ID.")


,raw_rows,rows_after_train_dedup,missing_cells,exact_duplicates_removed,exact_test_rows_preserved
train,150000,149575,0,425,0
test,50000,50000,0,0,5508


Training policy: keep first exact row in original order; preserve every test ID.


## 2. Reusable cleaning helpers

Dates support both formats observed across earlier file versions (`2021-01` and `Jan-21`). Text normalization is deliberately conservative: casing, whitespace, punctuation variants, and Unicode are standardized without inventing values.

In [3]:
def parse_rent_month(values: pd.Series) -> pd.Series:
    text = values.astype("string").str.strip()
    parsed = pd.Series(pd.NaT, index=values.index, dtype="datetime64[ns]")
    for fmt in ("%Y-%m", "%b-%y"):
        missing = parsed.isna()
        if not missing.any():
            break
        parsed.loc[missing] = pd.to_datetime(text.loc[missing], format=fmt, errors="coerce")
    if parsed.isna().any():
        bad = text.loc[parsed.isna()].drop_duplicates().tolist()[:10]
        raise ValueError(f"Unrecognized rental month values: {bad}")
    return parsed.dt.to_period("M").dt.to_timestamp()


def normalize_text(value):
    if pd.isna(value):
        return pd.NA
    value = unicodedata.normalize("NFKC", str(value)).strip().lower()
    value = re.sub(r"\s+", " ", value)
    return value


def normalize_block(value):
    value = normalize_text(value)
    if pd.isna(value):
        return pd.NA
    return re.sub(r"\s+", "", value)


def normalize_flat_type(value):
    value = normalize_text(value)
    if pd.isna(value):
        return pd.NA
    value = value.replace("-", " ")
    value = re.sub(r"\s+", " ", value).strip()
    aliases = {
        "1 room": "1-room", "2 room": "2-room", "3 room": "3-room",
        "4 room": "4-room", "5 room": "5-room", "executive": "executive",
    }
    return aliases.get(value, value.replace(" ", "-"))


def clean_number(values: pd.Series) -> pd.Series:
    return pd.to_numeric(
        values.astype("string").str.replace(r"[$,]", "", regex=True).str.strip(),
        errors="coerce",
    )


def month_index(values: pd.Series, base=pd.Timestamp("2021-01-01")) -> pd.Series:
    return (values.dt.year - base.year) * 12 + (values.dt.month - base.month)


FLAT_TYPE_ORDER = {
    "1-room": 1, "2-room": 2, "3-room": 3,
    "4-room": 4, "5-room": 5, "executive": 6,
}

## 3. Core-table preprocessing

The core table applies deterministic transformations only. `FURNISHED` and `FEE` are dropped because they are constant. The approval month becomes explicit trend and seasonal features. Text fields receive Unicode, whitespace, case, and flat-type normalization. Invalid lease ages become missing and receive a flag.

After normalization, exact training duplicates are audited again and removed. The original normalized `FLAT_MODEL` remains; no global rare-model column is created.


In [4]:
def prepare_core(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["RENT_DATE"] = parse_rent_month(out["RENT_APPROVAL_DATE"])
    out["RENT_YEAR"] = out["RENT_DATE"].dt.year.astype("int16")
    out["RENT_MONTH"] = out["RENT_DATE"].dt.month.astype("int8")
    out["RENT_QUARTER"] = out["RENT_DATE"].dt.quarter.astype("int8")
    out["MONTH_INDEX"] = month_index(out["RENT_DATE"]).astype("int16")
    out["MONTH_SIN"] = np.sin(2 * np.pi * out["RENT_MONTH"] / 12)
    out["MONTH_COS"] = np.cos(2 * np.pi * out["RENT_MONTH"] / 12)

    for column in ["TOWN", "STREET", "FLAT_MODEL"]:
        out[column] = out[column].map(normalize_text).astype("string")
    out["BLOCK"] = out["BLOCK"].map(normalize_block).astype("string")
    out["FLAT_TYPE"] = out["FLAT_TYPE"].map(normalize_flat_type).astype("string")

    out["PROPERTY_KEY"] = (
        out["TOWN"].fillna("") + "|" + out["BLOCK"].fillna("") + "|" + out["STREET"].fillna("")
    )
    out["TOWN_STREET"] = out["TOWN"].fillna("") + "|" + out["STREET"].fillna("")
    out["FLAT_TYPE_ORDER"] = out["FLAT_TYPE"].map(FLAT_TYPE_ORDER).astype("float32")

    lease_age = out["RENT_YEAR"] - pd.to_numeric(out["LEASE_COMMENCE_DATE"], errors="coerce")
    invalid_age = lease_age.lt(0) | lease_age.gt(100)
    out["LEASE_AGE_INVALID"] = invalid_age.astype("int8")
    out["LEASE_AGE_AT_RENT"] = lease_age.mask(invalid_age).astype("float32")
    out["LOG_FLOOR_AREA_SQM"] = np.log1p(pd.to_numeric(out["FLOOR_AREA_SQM"], errors="coerce"))

    return out.drop(columns=["RENT_APPROVAL_DATE", "FURNISHED", "FEE"])


train_normalized_before_dedup = prepare_core(train_raw_before_dedup)
normalized_feature_subset = [
    column for column in train_normalized_before_dedup.columns
    if column not in ["ROW_ID", target]
]
normalization_duplicate_audit = pd.DataFrame({
    "exact_duplicate_rows": [
        int(train_raw_before_dedup.drop(columns="ROW_ID").duplicated().sum()),
        int(train_normalized_before_dedup.drop(columns="ROW_ID").duplicated().sum()),
    ],
    "duplicate_feature_rows": [
        int(train_raw_before_dedup.duplicated(expected_features).sum()),
        int(train_normalized_before_dedup.duplicated(normalized_feature_subset).sum()),
    ],
}, index=["raw_train", "after_normalization_before_dedup"])
display(normalization_duplicate_audit)

train = prepare_core(train_raw)
test = prepare_core(test_raw)

post_normalization_exact_mask = train.drop(columns="ROW_ID").duplicated(keep="first")
post_normalization_exact_duplicates_removed = int(post_normalization_exact_mask.sum())
train = train.loc[~post_normalization_exact_mask].copy()

display(pd.DataFrame({
    "train_unique": train[["TOWN", "STREET", "FLAT_TYPE", "FLAT_MODEL"]].nunique(),
    "test_unique": test[["TOWN", "STREET", "FLAT_TYPE", "FLAT_MODEL"]].nunique(),
}))
print(f"Raw exact duplicates removed: {raw_exact_duplicates_removed:,}")
print(f"Additional exact duplicates after normalization: {post_normalization_exact_duplicates_removed:,}")
print(f"Invalid lease ages - train: {train['LEASE_AGE_INVALID'].sum()}, test: {test['LEASE_AGE_INVALID'].sum()}")


,exact_duplicate_rows,duplicate_feature_rows
raw_train,425,15547
after_normalization_before_dedup,641,21974


,train_unique,test_unique
TOWN,26,26
STREET,576,573
FLAT_TYPE,6,6
FLAT_MODEL,20,19


Raw exact duplicates removed: 425
Additional exact duplicates after normalization: 216
Invalid lease ages - train: 1, test: 0


## 4. Auxiliary feature audit only

The following HDB, amenity, COE, and stock cells remain as reproducible feature audits, but they do not modify the core `train` or `test` tables. The main auxiliary pipeline lives in `04_auxiliary_data_preprocessing.ipynb`.


In [5]:
hdb = pd.read_csv(AUX_DIR / "sg-hdb-block.csv", dtype={"POSTAL_CODE": "string"})
for column in ["TOWN", "STREET"]:
    hdb[column] = hdb[column].map(normalize_text).astype("string")
hdb["BLOCK"] = hdb["BLOCK"].map(normalize_block).astype("string")

join_keys = ["TOWN", "BLOCK", "STREET"]
assert not hdb.duplicated(join_keys).any(), "HDB block join keys must be unique"

hdb["MAX_FLOOR"] = pd.to_numeric(hdb["MAX_FLOOR"], errors="coerce").replace(-1, np.nan)
hdb["YEAR_COMPLETED"] = pd.to_numeric(hdb["YEAR_COMPLETED"], errors="coerce").replace(-1, np.nan)
hdb["POSTAL_CODE"] = hdb["POSTAL_CODE"].replace({"NIL": pd.NA, "nil": pd.NA})
hdb["POSTAL_CODE"] = hdb["POSTAL_CODE"].str.zfill(6)
hdb["POSTAL_SECTOR"] = hdb["POSTAL_CODE"].str[:2]

hdb_features = hdb.rename(columns={
    "POSTAL_CODE": "HDB_POSTAL_CODE",
    "POSTAL_SECTOR": "HDB_POSTAL_SECTOR",
    "LATITUDE": "HDB_LATITUDE",
    "LONGITUDE": "HDB_LONGITUDE",
    "MAX_FLOOR": "HDB_MAX_FLOOR",
    "YEAR_COMPLETED": "HDB_YEAR_COMPLETED",
    "SUBZONE": "HDB_SUBZONE",
    "PLANNING_AREA": "HDB_PLANNING_AREA",
    "REGION": "HDB_REGION",
})


def nearest_distance_and_count(
    origins: pd.DataFrame,
    destinations: pd.DataFrame,
    radius_km: float = 1.0,
) -> tuple[np.ndarray, np.ndarray]:
    origin_rad = np.radians(origins[["HDB_LATITUDE", "HDB_LONGITUDE"]].to_numpy(float))
    destination_rad = np.radians(destinations[["LATITUDE", "LONGITUDE"]].to_numpy(float))
    tree = BallTree(destination_rad, metric="haversine")
    nearest_rad, _ = tree.query(origin_rad, k=1)
    counts = tree.query_radius(origin_rad, r=radius_km / EARTH_RADIUS_KM, count_only=True)
    return nearest_rad[:, 0] * EARTH_RADIUS_KM, counts.astype("int16")


mrt = pd.read_csv(AUX_DIR / "sg-mrt-stations.csv")
mrt["CODE_CLEAN"] = mrt["CODE"].map(normalize_text)
mrt["CODE_PREFIX"] = mrt["CODE_CLEAN"].str.extract(r"^([a-z]+)", expand=False)
pre2021_prefixes = {"ns", "ew", "cg", "ne", "cc", "ce", "dt"}
post2021_or_future_codes = {"ns6", "ne18", "cc30", "cc31", "cc32", "dt4", "dt36", "dt37"}
mrt = mrt.loc[
    mrt["STATUS"].map(normalize_text).eq("open")
    & mrt["CODE_PREFIX"].isin(pre2021_prefixes)
    & ~mrt["CODE_CLEAN"].isin(post2021_or_future_codes)
].copy()
# Multiple line codes can describe the same physical interchange.
mrt["STATION_KEY"] = (
    mrt["NAME"].map(normalize_text).astype("string") + "|" +
    mrt["LATITUDE"].round(5).astype(str) + "|" + mrt["LONGITUDE"].round(5).astype(str)
)
mrt = mrt.drop_duplicates("STATION_KEY")

malls = pd.read_csv(AUX_DIR / "sg-shopping-malls.csv").drop_duplicates(["LATITUDE", "LONGITUDE"])
schools = pd.read_csv(AUX_DIR / "sg-schools.csv")
primary_mask = schools["MAINLEVEL_CODE"].astype("string").str.lower().str.contains(
    r"primary|p1", regex=True, na=False
)
primary_schools = schools.loc[primary_mask].drop_duplicates(["LATITUDE", "LONGITUDE"])

geo = hdb_features[join_keys + ["HDB_LATITUDE", "HDB_LONGITUDE"]].copy()
for name, destinations in {
    "PRE2021_MRT": mrt,
    "MALL": malls,
    "PRIMARY_SCHOOL": primary_schools,
}.items():
    distance, count = nearest_distance_and_count(geo, destinations, radius_km=1.0)
    geo[f"DIST_TO_NEAREST_{name}_KM"] = distance.astype("float32")
    geo[f"{name}_COUNT_WITHIN_1KM"] = count

hdb_features = hdb_features.merge(geo.drop(columns=["HDB_LATITUDE", "HDB_LONGITUDE"]), on=join_keys, validate="one_to_one")


def merge_hdb_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.merge(hdb_features, on=join_keys, how="left", validate="many_to_one", indicator="_HDB_MERGE")
    out["HDB_MATCHED"] = out["_HDB_MERGE"].eq("both").astype("int8")
    out = out.drop(columns="_HDB_MERGE")
    building_age = out["RENT_YEAR"] - out["HDB_YEAR_COMPLETED"]
    invalid = building_age.lt(0) | building_age.gt(100)
    out["BUILDING_AGE_INVALID"] = invalid.fillna(False).astype("int8")
    out["BUILDING_AGE_AT_RENT"] = building_age.mask(invalid).astype("float32")
    return out


train_hdb_preview = merge_hdb_features(train)
test_hdb_preview = merge_hdb_features(test)

hdb_join_rates = {
    "train": float(train_hdb_preview["HDB_MATCHED"].mean()),
    "test": float(test_hdb_preview["HDB_MATCHED"].mean()),
}
display(pd.Series(hdb_join_rates, name="HDB join rate"))
assert min(hdb_join_rates.values()) >= 0.99, "Unexpectedly low HDB join coverage"

distance_columns = [column for column in train_hdb_preview.columns if column.startswith("DIST_TO_NEAREST_")]
display(train_hdb_preview[distance_columns].describe().T[["mean", "50%", "max"]])

train   1.0000
test    1.0000
Name: HDB join rate, dtype: float64

,mean,50%,max
DIST_TO_NEAREST_PRE2021_MRT_KM,0.7221,0.6347,3.5147
DIST_TO_NEAREST_MALL_KM,0.8104,0.7323,3.2024
DIST_TO_NEAREST_PRIMARY_SCHOOL_KM,0.4267,0.3747,3.3092


## 5. Lagged COE market features

COE auction results are aggregated within each month and category, then shifted by one calendar month. This avoids using an auction result that might not have been known when a rental was approved. Category C is retained even though it is less directly related to HDB demand; its value can be judged during feature ablation.

In [6]:
coe = pd.read_csv(AUX_DIR / "sg-coe-prices.csv")
coe["CATEGORY"] = coe["CATEGORY"].map(normalize_text).str.upper()
coe["COE_DATE"] = pd.to_datetime(
    coe["YEAR"].astype(str) + "-" + coe["MONTH"].astype(str),
    format="%Y-%b",
    errors="raise",
).dt.to_period("M").dt.to_timestamp()

for column in ["QUOTA_PREMIUM", "PREVAILING_QUOTA_PREMIUM", "QUOTA", "BIDS_RECEIVED"]:
    coe[column] = clean_number(coe[column])
coe["BID_TO_QUOTA_RATIO"] = coe["BIDS_RECEIVED"] / coe["QUOTA"].replace(0, np.nan)

coe_monthly_long = coe.groupby(["COE_DATE", "CATEGORY"], as_index=False).agg(
    PREMIUM_MEAN=("QUOTA_PREMIUM", "mean"),
    PREVAILING_PREMIUM_MEAN=("PREVAILING_QUOTA_PREMIUM", "mean"),
    QUOTA_SUM=("QUOTA", "sum"),
    BIDS_SUM=("BIDS_RECEIVED", "sum"),
    BID_TO_QUOTA_RATIO=("BID_TO_QUOTA_RATIO", "mean"),
)

coe_monthly = coe_monthly_long.pivot(index="COE_DATE", columns="CATEGORY")
coe_monthly.columns = [f"COE_{category}_{metric}" for metric, category in coe_monthly.columns]
coe_monthly = coe_monthly.sort_index()
full_months = pd.date_range(coe_monthly.index.min(), coe_monthly.index.max(), freq="MS")
coe_monthly = coe_monthly.reindex(full_months).shift(1)
coe_monthly.columns = [f"{column}_LAG1" for column in coe_monthly.columns]
coe_monthly.index.name = "RENT_DATE"
coe_monthly = coe_monthly.reset_index()

display(coe_monthly.tail(3))

,RENT_DATE,COE_A_PREMIUM_MEAN_LAG1,COE_B_PREMIUM_MEAN_LAG1,COE_C_PREMIUM_MEAN_LAG1,COE_E_PREMIUM_MEAN_LAG1,COE_A_PREVAILING_PREMIUM_MEAN_LAG1,COE_B_PREVAILING_PREMIUM_MEAN_LAG1,COE_C_PREVAILING_PREMIUM_MEAN_LAG1,COE_E_PREVAILING_PREMIUM_MEAN_LAG1,COE_A_QUOTA_SUM_LAG1,COE_B_QUOTA_SUM_LAG1,COE_C_QUOTA_SUM_LAG1,COE_E_QUOTA_SUM_LAG1,COE_A_BIDS_SUM_LAG1,COE_B_BIDS_SUM_LAG1,COE_C_BIDS_SUM_LAG1,COE_E_BIDS_SUM_LAG1,COE_A_BID_TO_QUOTA_RATIO_LAG1,COE_B_BID_TO_QUOTA_RATIO_LAG1,COE_C_BID_TO_QUOTA_RATIO_LAG1,COE_E_BID_TO_QUOTA_RATIO_LAG1
65,2026-06-01,"124,509.5000","127,868.5000","89,851.0000","128,850.0000","112,324.0000","114,577.0000","77,884.0000",0.0000,2540,1752,585,510,4354,2801,979,915,1.7172,1.5995,1.6734,1.7945
66,2026-07-01,"124,928.0000","125,245.5000","93,500.5000","129,001.0000","118,357.0000","121,218.0000","82,868.0000",0.0000,2497,1772,584,523,3844,2502,843,984,1.5397,1.4118,1.4431,1.8815
67,2026-08-01,"127,500.0000","130,389.5000","94,444.5000","129,886.0000","123,315.0000","124,705.0000","88,368.0000",0.0000,2489,1738,597,540,3469,2865,872,902,1.3938,1.6486,1.4609,1.6714


## 6. Lagged stock-market feature audit

Stock returns are calculated per symbol and summarized by month. Missing monthly returns are excluded from `STOCK_POSITIVE_SHARE`; they are not counted as non-positive returns. These features remain outside the core output.


In [7]:
stocks = pd.read_csv(AUX_DIR / "sg-stock-prices.csv")
stocks["DATE"] = pd.to_datetime(stocks["DATE"], errors="coerce")
stocks["SYMBOL"] = stocks["SYMBOL"].astype("string")
for column in ["OPEN", "HIGH", "LOW", "CLOSE", "ADJUSTED_CLOSE", "VOLUME"]:
    stocks[column] = pd.to_numeric(stocks[column], errors="coerce")
stocks = stocks.dropna(subset=["DATE", "SYMBOL", "ADJUSTED_CLOSE"]).sort_values(["SYMBOL", "DATE"])
stocks = stocks.loc[stocks["ADJUSTED_CLOSE"].gt(0)].copy()
stocks["STOCK_MONTH"] = stocks["DATE"].dt.to_period("M").dt.to_timestamp()
stocks["DAILY_RETURN"] = stocks.groupby("SYMBOL", observed=True)["ADJUSTED_CLOSE"].pct_change(fill_method=None)
stocks["DAILY_RETURN_CLEAN"] = stocks["DAILY_RETURN"].mask(stocks["DAILY_RETURN"].abs().gt(0.50))

daily_vol = stocks.groupby(["SYMBOL", "STOCK_MONTH"], observed=True)["DAILY_RETURN_CLEAN"].std().rename("DAILY_VOL")
month_end = stocks.groupby(["SYMBOL", "STOCK_MONTH"], observed=True).tail(1).copy()
month_end["MONTHLY_RETURN"] = month_end.groupby("SYMBOL", observed=True)["ADJUSTED_CLOSE"].pct_change(fill_method=None)
month_end = month_end.merge(daily_vol.reset_index(), on=["SYMBOL", "STOCK_MONTH"], how="left", validate="one_to_one")

# Cross-sectional winsorization is performed independently within each month.
def positive_share(values: pd.Series) -> float:
    valid = values.dropna()
    return valid.gt(0).mean() if len(valid) else np.nan


def winsorize_month(series: pd.Series) -> pd.Series:
    if series.notna().sum() < 5:
        return series
    lower, upper = series.quantile([0.01, 0.99])
    return series.clip(lower, upper)


month_end["MONTHLY_RETURN_WINSORIZED"] = month_end.groupby("STOCK_MONTH", observed=True)["MONTHLY_RETURN"].transform(winsorize_month)
market_monthly = month_end.groupby("STOCK_MONTH", as_index=False, observed=True).agg(
    STOCK_RETURN_MEDIAN=("MONTHLY_RETURN", "median"),
    STOCK_RETURN_MEAN_WINSORIZED=("MONTHLY_RETURN_WINSORIZED", "mean"),
    STOCK_RETURN_STD=("MONTHLY_RETURN", "std"),
    STOCK_POSITIVE_SHARE=("MONTHLY_RETURN", positive_share),
    STOCK_DAILY_VOL_MEDIAN=("DAILY_VOL", "median"),
    STOCK_ACTIVE_SYMBOL_COUNT=("SYMBOL", "nunique"),
    STOCK_RETURN_SYMBOL_COUNT=("MONTHLY_RETURN", "count"),
)
market_monthly = market_monthly.set_index("STOCK_MONTH").sort_index()
full_months = pd.date_range(market_monthly.index.min(), market_monthly.index.max(), freq="MS")
market_monthly = market_monthly.reindex(full_months).shift(1)
market_monthly.columns = [f"{column}_LAG1" for column in market_monthly.columns]
market_monthly.index.name = "RENT_DATE"
market_monthly = market_monthly.reset_index()

display(market_monthly.tail(3))

,RENT_DATE,STOCK_RETURN_MEDIAN_LAG1,STOCK_RETURN_MEAN_WINSORIZED_LAG1,STOCK_RETURN_STD_LAG1,STOCK_POSITIVE_SHARE_LAG1,STOCK_DAILY_VOL_MEDIAN_LAG1,STOCK_ACTIVE_SYMBOL_COUNT_LAG1,STOCK_RETURN_SYMBOL_COUNT_LAG1
64,2026-05-01,0.0250,0.0561,0.2132,0.6623,0.0191,77.0000,77.0000
65,2026-06-01,0.0000,0.0309,0.1914,0.4805,0.0228,77.0000,77.0000
66,2026-07-01,-0.0201,-0.0573,0.1473,0.3506,0.0172,77.0000,77.0000


## 7. Finalize the core schema and audit duplicates

Only the deterministic core fields enter the saved datasets. Categorical missing values receive an explicit token. The final audit reports exact duplicates and duplicate feature vectors after normalization, making the effect of preprocessing visible.


In [8]:
categorical_features = [
    "TOWN", "BLOCK", "STREET", "FLAT_TYPE", "FLAT_MODEL", "PROPERTY_KEY", "TOWN_STREET",
]
for column in categorical_features:
    train[column] = train[column].astype("string").fillna("__missing__")
    test[column] = test[column].astype("string").fillna("__missing__")

train = train.sort_values("ROW_ID").reset_index(drop=True)
test = test.sort_values("ROW_ID").reset_index(drop=True)
feature_columns = test.columns.tolist()
train = train[feature_columns + [target]]

non_feature_columns = ["ROW_ID", "RENT_DATE"]
model_features = [column for column in feature_columns if column not in non_feature_columns]
duplicate_feature_subset = [column for column in feature_columns if column != "ROW_ID"]

postprocessing_audit = pd.DataFrame({
    "rows": [len(train), len(test)],
    "exact_duplicates": [
        int(train.drop(columns="ROW_ID").duplicated().sum()),
        int(test.drop(columns="ROW_ID").duplicated().sum()),
    ],
    "duplicate_feature_rows": [
        int(train.duplicated(duplicate_feature_subset).sum()),
        int(test.duplicated(duplicate_feature_subset).sum()),
    ],
}, index=["train_core", "test_core"])
display(postprocessing_audit)

assert train["ROW_ID"].is_unique and train["ROW_ID"].is_monotonic_increasing
assert test["ROW_ID"].tolist() == list(range(len(test_raw)))
assert train.drop(columns=[target]).columns.tolist() == test.columns.tolist()
assert train.drop(columns="ROW_ID").duplicated().sum() == 0
assert "FLAT_MODEL_GROUPED" not in train.columns
assert "RENT_PER_SQM" not in train.columns
assert not any(column.startswith(("HDB_", "COE_", "STOCK_", "DIST_TO_")) for column in model_features)
train_numeric = train.select_dtypes(include="number").to_numpy(dtype=float, na_value=np.nan)
test_numeric = test.select_dtypes(include="number").to_numpy(dtype=float, na_value=np.nan)
assert not np.isinf(train_numeric).any() and not np.isinf(test_numeric).any()

summary = pd.DataFrame({
    "rows": [len(train), len(test)],
    "columns": [train.shape[1], test.shape[1]],
    "model_features": [len(model_features), len(model_features)],
    "numeric_missing_cells": [
        int(train.select_dtypes(include=np.number).isna().sum().sum()),
        int(test.select_dtypes(include=np.number).isna().sum().sum()),
    ],
}, index=["train_preprocessed", "test_preprocessed"])
display(summary)


,rows,exact_duplicates,duplicate_feature_rows
train_core,149359,0,21333
test_core,50000,7764,7764


,rows,columns,model_features,numeric_missing_cells
train_preprocessed,149359,22,19,1
test_preprocessed,50000,21,19,0


## 8. Save processed core artifacts

The metadata records the feature contract, both exact-deduplication stages, and the remaining feature-duplicate count. `ROW_ID` stays as an alignment key but never enters the model matrix.


In [9]:
metadata = {
    "target": target,
    "row_id": "ROW_ID",
    "date_column": "RENT_DATE",
    "non_feature_columns": non_feature_columns,
    "model_features": model_features,
    "categorical_features": categorical_features,
    "train_rows": len(train),
    "test_rows": len(test),
    "duplicate_audit": {
        "raw_train_rows": raw_train_rows,
        "raw_exact_duplicates_removed": raw_exact_duplicates_removed,
        "normalized_exact_duplicates_before_dedup": int(normalization_duplicate_audit.loc["after_normalization_before_dedup", "exact_duplicate_rows"]),
        "normalized_feature_duplicates_before_dedup": int(normalization_duplicate_audit.loc["after_normalization_before_dedup", "duplicate_feature_rows"]),
        "post_normalization_exact_duplicates_removed": post_normalization_exact_duplicates_removed,
        "final_exact_duplicates": int(train.drop(columns="ROW_ID").duplicated().sum()),
        "final_duplicate_feature_rows": int(train.duplicated(duplicate_feature_subset).sum()),
    },
    "preprocessing_decisions": {
        "target_outliers_removed": False,
        "constant_columns_dropped": ["FURNISHED", "FEE"],
        "text_normalization": "NFKC, lowercase, trim and collapse whitespace; normalize block and flat-type variants",
        "calendar_features": ["RENT_YEAR", "RENT_MONTH", "RENT_QUARTER", "MONTH_INDEX", "MONTH_SIN", "MONTH_COS"],
        "location_keys": ["PROPERTY_KEY", "TOWN_STREET"],
        "invalid_ages": "set to missing and flagged",
        "floor_area": "retain raw value and add log1p transformation",
        "rare_categories": "not grouped in preprocessing; OneHotEncoder learns infrequent categories inside each training fold",
        "auxiliary_features": "excluded from core outputs",
        "target_derived_features": "forbidden",
    },
}

if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    train.to_csv(OUTPUT_DIR / "train_preprocessed.csv", index=False, date_format="%Y-%m-%d")
    test.to_csv(OUTPUT_DIR / "test_preprocessed.csv", index=False, date_format="%Y-%m-%d")
    with (OUTPUT_DIR / "preprocessing_metadata.json").open("w", encoding="utf-8") as handle:
        json.dump(metadata, handle, ensure_ascii=False, indent=2)
    print(f"Saved processed files to: {OUTPUT_DIR.relative_to(PROJECT_ROOT)}")
    for path in sorted(OUTPUT_DIR.glob("*")):
        print(f"- {path.name}: {path.stat().st_size / 1_048_576:.2f} MiB")


Saved processed files to: data/processed/core
- preprocessing_metadata.json: 0.00 MiB
- test_preprocessed.csv: 10.21 MiB
- train_preprocessed.csv: 31.19 MiB


## 9. Modeling handoff

- Sort rows by `RENT_DATE` and evaluate expanding chronological folds.
- Exclude `ROW_ID` and `RENT_DATE` from model inputs; keep the derived calendar fields.
- Fit numeric imputation, scaling, one-hot categories, and rare-category pooling inside each fold's historical training rows.
- Keep only `FLAT_MODEL`; do not reintroduce `FLAT_MODEL_GROUPED`.
- Compare this core dataset with the HDB + conservative MRT dataset using the same temporal windows.
- Refit the selected pipeline on all labelled rows, then map test predictions back to sample-submission IDs.

Core preprocessing never uses the target for filtering, encoding, imputation, or feature construction, except that exact rows include the target when determining whether records are identical.
